# Setup

In [2]:
# import basic packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import PIL
import torch
from utils import embedding_dataframe_construction

# Useful Functions

In [3]:
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection
from torch.utils.data import DataLoader
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

image_processor = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
vision_model = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-base-patch32").to(device)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0

In [4]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


GPU Available: True
Device Name: NVIDIA GeForce MX250


# Import Dataset

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\pedro\.cache\kagglehub\datasets\sumn2u\garbage-classification-v2\versions\12


In [7]:
from datasets import load_dataset
dataset = load_dataset(path + "/standardized_384", split="train")

labels = dataset.features["label"].names
print(f"Labels: {labels}")

Resolving data files:   0%|          | 0/12259 [00:00<?, ?it/s]

Labels: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']


In [8]:
#filtering by trash labels 
trash_filter = dataset.filter(lambda x: x['label'] == 9)

In [9]:
trash_embedding_dataframe = embedding_dataframe_construction(trash_filter, trash_filter['label'], vision_model)

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [01:05<00:00,  4.34s/it]


In [11]:
trash_embedding_mean = trash_embedding_dataframe.drop(columns = 'Class').mean()

In [13]:
#calculate normalized vector 
norm_embedding = trash_embedding_mean / np.linalg.norm(trash_embedding_mean, axis=-1, keepdims=True)

In [14]:
candidate_phrases = [
    "a photo of trash",                                                    # 1. baseline, current label
    "a photo of garbage",                                                  # 2. synonym baseline
    "a photo of waste",                                                    # 3. synonym baseline
    "a photo of non-recyclable waste",                                     # 4. your hypothesis
    "a photo of general waste",                                            # 5. common bin-labeling term
    "a photo of residual waste",                                           # 6. common bin-labeling term (EU)
    "a photo of landfill waste",                                           # 7. destination-based descriptor
    "a photo of unsortable waste",                                         # 8. process-based descriptor
    "a photo of mixed waste",                                              # 9. composition-based descriptor
    "a photo of household trash",                                         # 10. context added
    "a photo of miscellaneous garbage",                                    # 11. adjective added
    "a photo of trash that cannot be recycled",                            # 12. explanatory phrase
    "a photo of waste that goes to landfill",                              # 13. explanatory phrase
    "a photo of items unfit for recycling",                                # 14. rephrased explanatory
    "a close-up photo of non-recyclable household waste",                  # 15. added visual framing
    "a photo of disposable items like diapers and wrappers",               # 16. concrete examples
    "a photo of styrofoam, wrappers, and other non-recyclable trash",      # 17. concrete material examples
    "a photo of dirty, contaminated waste unsuitable for recycling",       # 18. texture/condition descriptor
    "a photo of a pile of non-recyclable household garbage",               # 19. composition + context
    "a high-resolution photo of assorted non-recyclable waste items",      # 20. full complexity
]


# Calculating Text Embeddings

In [15]:
from transformers import CLIPTextModelWithProjection, AutoTokenizer

text_model = CLIPTextModelWithProjection.from_pretrained("openai/clip-vit-base-patch32").to(device)
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")


tokens = tokenizer(candidate_phrases, padding = True, return_tensors = 'pt').to(device)
with torch.no_grad():
    outputs =  text_model(**tokens)

text_embeddings = outputs.text_embeds
text_embeddings = text_embeddings/text_embeddings.norm(dim=-1, keepdim=True)  

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_m

# Computing Cosine Similarity

In [16]:
torch_image_vector =  torch.from_numpy(norm_embedding.to_numpy()).float()

In [17]:
cos_sim = text_embeddings.to(device) @ torch_image_vector.to(device)

In [20]:
ranked = sorted(zip(candidate_phrases, cos_sim.tolist()), key=lambda x: x[1], reverse=True)
for rank, (phrase, score) in enumerate(ranked, start=1):
    print(f"{rank:2d}. {phrase:65s} sim={score:.4f}")

 1. a photo of disposable items like diapers and wrappers             sim=0.3109
 2. a photo of items unfit for recycling                              sim=0.3098
 3. a photo of non-recyclable waste                                   sim=0.3018
 4. a photo of residual waste                                         sim=0.2971
 5. a photo of waste                                                  sim=0.2961
 6. a photo of styrofoam, wrappers, and other non-recyclable trash    sim=0.2936
 7. a high-resolution photo of assorted non-recyclable waste items    sim=0.2898
 8. a photo of miscellaneous garbage                                  sim=0.2889
 9. a photo of mixed waste                                            sim=0.2887
10. a photo of dirty, contaminated waste unsuitable for recycling     sim=0.2881
11. a close-up photo of non-recyclable household waste                sim=0.2880
12. a photo of trash that cannot be recycled                          sim=0.2876
13. a photo of unsortable wa